# CONUS Datetime Parsing and Basic Analytics

This notebook is a focused walkthrough of datetime parsing using real U.S. electricity demand data.

Dataset: conus_hourly_demand.csv (hourly, UTC timestamps).

## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
print("pandas", pd.__version__)

## 2. Load the CSV and inspect schema

In [ ]:
csv_path = Path("conus_hourly_demand.csv")
if not csv_path.exists():
    raise FileNotFoundError(f"Missing file: {csv_path.resolve()}")

raw = pd.read_csv(csv_path, na_values=["MISSING", "EMPTY"])
print("shape:", raw.shape)
print("columns:", raw.columns.tolist())
raw.head()

## 3. Parse datetime column as UTC (core datetime step)

In [ ]:
df = raw.copy()
df["timestamp_utc"] = pd.to_datetime(df["date_time"], errors="coerce", utc=True)

bad_ts = df["timestamp_utc"].isna().sum()
print("parsed timestamps:", len(df) - bad_ts)
print("invalid timestamps:", bad_ts)

df[["date_time", "timestamp_utc"]].head()

## 4. Datetime quality checks

In [ ]:
df["cleaned_demand_mw"] = pd.to_numeric(df["cleaned demand (MW)"], errors="coerce")

work = df.dropna(subset=["timestamp_utc", "cleaned_demand_mw"]).copy()
work = work.sort_values("timestamp_utc")

dup_count = work["timestamp_utc"].duplicated().sum()
is_monotonic = work["timestamp_utc"].is_monotonic_increasing

dt_diffs = work["timestamp_utc"].diff().dropna()
mode_step = dt_diffs.mode().iloc[0]
hourly_like_ratio = (dt_diffs == pd.Timedelta(hours=1)).mean()

print("valid rows:", len(work))
print("duplicate timestamps:", int(dup_count))
print("monotonic increasing:", bool(is_monotonic))
print("most common step:", mode_step)
print("exact 1-hour step ratio:", round(float(hourly_like_ratio), 6))

## 5. Set datetime index and create local-time features

In [ ]:
ts = work.set_index("timestamp_utc").copy()
ts.index.name = "timestamp_utc"

# Convert to a local timezone for calendar-style analytics
ts["timestamp_central"] = ts.index.tz_convert("America/Chicago")

ts["year"] = ts["timestamp_central"].dt.year
ts["month"] = ts["timestamp_central"].dt.month
ts["weekday"] = ts["timestamp_central"].dt.day_name()
ts["hour"] = ts["timestamp_central"].dt.hour
ts["is_weekend"] = ts["timestamp_central"].dt.dayofweek >= 5

ts[["cleaned_demand_mw", "timestamp_central", "year", "month", "weekday", "hour", "is_weekend"]].head()

## 6. Basic datetime slices

In [ ]:
july_2019_weekdays = ts[(ts["year"] == 2019) & (ts["month"] == 7) & (~ts["is_weekend"])].copy()
print("July 2019 weekday rows:", len(july_2019_weekdays))
july_2019_weekdays[["cleaned_demand_mw", "timestamp_central"]].head()

## 7. Hourly, weekday, and monthly demand patterns

In [ ]:
hourly_profile = ts.groupby("hour")["cleaned_demand_mw"].mean()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_profile = ts.groupby("weekday")["cleaned_demand_mw"].mean().reindex(weekday_order)
monthly_profile = ts.groupby("month")["cleaned_demand_mw"].mean().reindex(range(1, 13))

fig, axes = plt.subplots(3, 1, figsize=(11, 12))

hourly_profile.plot(ax=axes[0], marker="o", title="Average Demand by Hour (America/Chicago)")
axes[0].set_xlabel("Hour")
axes[0].set_ylabel("MW")
axes[0].grid(True, alpha=0.3)

weekday_profile.plot(ax=axes[1], marker="o", title="Average Demand by Weekday")
axes[1].set_xlabel("Weekday")
axes[1].set_ylabel("MW")
axes[1].grid(True, alpha=0.3)

monthly_profile.plot(ax=axes[2], marker="o", title="Average Demand by Month")
axes[2].set_xlabel("Month")
axes[2].set_ylabel("MW")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. DST behavior after timezone conversion

The UTC index is continuous, but local day counts can be 23, 24, or 25 around DST transitions.

In [ ]:
local_series = ts["cleaned_demand_mw"].copy()
local_series.index = local_series.index.tz_convert("America/Chicago")

counts_by_local_day = local_series.resample("D").count()

# Drop boundary days that can be partial due to dataset start/end cutoffs.
inner_counts = counts_by_local_day.iloc[1:-1]
dst_days = inner_counts[inner_counts != 24]

print("Number of non-24-hour local days (interior only):", len(dst_days))
print(dst_days.head(10))

ax = dst_days.plot(kind="bar", figsize=(12, 4), title="Local-Day Hour Counts Around DST")
ax.set_ylabel("Hourly observations")
ax.axhline(24, linestyle="--")
plt.tight_layout()
plt.show()